# s5figure.ipynb

Manuscript analysis and plotting notebook.

Run the configuration cell first. Input data and fonts are not bundled.
See the repository README and data requirements before executing.


In [ ]:
from pathlib import Path
import sys as _sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'plot_config.py').is_file()
     and (p / 'config.example.json').is_file()),
    None,
)
if _repo_root is None:
    raise RuntimeError('Run this notebook from within the plotting repository.')
if str(_repo_root) not in _sys.path:
    _sys.path.insert(0, str(_repo_root))
from plot_config import configured_path


In [ ]:
import numpy as np  
import matplotlib.pyplot as plt  
import matplotlib.patches as patches 
import xarray as xr
import pandas as pd
import cartopy.crs as ccrs  
import cartopy.feature as cfeature
import cmaps
from cartopy.util import add_cyclic_point
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
from tqdm import tqdm
import gsw

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [ ]:
import pandas as pd
import numpy as np
import gsw

# ======================================================
# 1. 读取并整理站点数据
# ======================================================
df = pd.read_csv(configured_path('bats_profiles_file'))

df = df.rename(columns={
    "Depth": "depth",
    "Temp_x": "temp",
    "Sal1_x": "sal"
})

df[["depth", "temp", "sal"]] = df[["depth", "temp", "sal"]].astype(float)

# ======================================================
# 2. 单剖面 MLD（严格复刻你原定义）
# ======================================================
def calc_mld_sigma0_station(
    depth, temp, sal,
    lat=31.67, lon=295.83,
    ref_depth=10.0,
    threshold=0.03
):
    mask = np.isfinite(depth) & np.isfinite(temp) & np.isfinite(sal)
    depth = depth[mask]
    temp  = temp[mask]
    sal   = sal[mask]

    if len(depth) < 3:
        return np.nan

    idx = np.argsort(depth)
    depth = depth[idx]
    temp  = temp[idx]
    sal   = sal[idx]

    if ref_depth < depth.min() or ref_depth > depth.max():
        return np.nan

    # 压力
    p = gsw.p_from_z(-depth, lat)

    # σ0
    SA = gsw.SA_from_SP(sal, p, lon, lat)
    sigma0 = gsw.sigma0(SA, temp)

    # 10 m 参考层（插值）
    sigma0_ref = np.interp(ref_depth, depth, sigma0)

    delta = np.abs(sigma0 - sigma0_ref)

    valid = (depth > ref_depth) & (delta >= threshold)
    if not np.any(valid):
        return np.nan

    i2 = np.where(valid)[0][0]
    i1 = i2 - 1
    if i1 < 0:
        return np.nan

    d1, d2 = depth[i1], depth[i2]
    r1, r2 = delta[i1], delta[i2]

    if r2 == r1:
        mld = d2
    else:
        frac = (threshold - r1) / (r2 - r1)
        mld = d1 + frac * (d2 - d1)

    # 新增条件：必须大于 ref_depth
    if mld <= ref_depth:
        return np.nan

    return mld


# ======================================================
# 3. 按日期循环，计算完整时间序列
# ======================================================
records = []

for date, g in df.groupby("yyyymmdd"):
    mld = calc_mld_sigma0_station(
        g["depth"].values,
        g["temp"].values,
        g["sal"].values
    )

    records.append({
        "yyyymmdd": pd.to_datetime(str(date)),
        "MLD_sigma0_003": mld
    })

mld_df = pd.DataFrame(records).sort_values("yyyymmdd")

print(mld_df.head())


In [ ]:
mld_df = mld_df.set_index("yyyymmdd")
mld_df.index.name = "time"

mld_da = xr.DataArray(
    mld_df["MLD_sigma0_003"],
    dims=("time",),
    coords={"time": mld_df.index},
    name="MLD_sigma0_003",
    attrs={
        "long_name": "Mixed Layer Depth",
        "definition": "|Δσ₀| ≥ 0.03 kg m⁻³ (ref = 10 m, interpolated)",
        "units": "m",
        "density_type": "sigma0 (TEOS-10)",
        "reference_depth": 10.0,
        "threshold": 0.03,
        "method": "station profile, linear interpolation",
    }
)
out_nc = "BAT_MLD_sigma0_003.nc"
#mld_da.to_netcdf(out_nc)

In [ ]:
# 1. 按月初重采样求平均
monthly_ds = mld_da.resample(time='1MS').mean()

# 2. 将时间坐标向后移动15天，使时间点落在月中
monthly_ds['time'] = monthly_ds['time'] + pd.Timedelta(days=14)
#monthly_ds

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)
#ds

In [ ]:
def four_neighbor_stats(ds: xr.Dataset, var_name: str, target_lat, target_lon):
    lat = ds.lat.values
    lon = ds.lon.values

    # 找到距离目标点最近网格索引
    ilat = np.argmin(np.abs(lat - target_lat))
    ilon = np.argmin(np.abs(lon - target_lon))

    # 取左右/上下各一格，组成2×2四点
    lat_inds = []
    for di in (-1, 0):
        idx = ilat + di
        if 0 <= idx < len(lat):
            lat_inds.append(idx)

    lon_inds = []
    for dj in (-1, 0):
        idx = ilon + dj
        if 0 <= idx < len(lon):
            lon_inds.append(idx)

    # 截取2×2子区域，保留time维度
    sub_da = ds[var_name].isel(lat=lat_inds, lon=lon_inds)

    # 四点均值（原需求）
    mean_da = sub_da.mean(dim=["lat", "lon"], skipna=True)
    # 四点标准差（用于shading误差填色）
    std_da = sub_da.std(dim=["lat", "lon"], skipna=True)

    # 统一赋值目标经纬度坐标，和interp输出格式对齐
    mean_da = mean_da.assign_coords(lat=target_lat, lon=target_lon)
    std_da = std_da.assign_coords(lat=target_lat, lon=target_lon)

    return mean_da, std_da

In [ ]:
lat0, lon0 = 31.67, 295.83
# 调用
mld_mean, mld_std = four_neighbor_stats(ds, "MLD_pred", lat0, lon0)

In [ ]:
mld_mean = mld_mean.sel(time=slice('2010-1','2020-12'))
mld_std = mld_std.sel(time=slice('2010-1','2020-12'))
monthly_ds = monthly_ds.sel(time=slice('2010-1','2020-12'))

# 【修改1】mask第二个条件改为 mld_mean，原mld_pred变量不存在
mask = (~np.isnan(monthly_ds)) & (~np.isnan(mld_mean))

# 【修改2】同步筛选std，保证时间对齐
da1_clean_1 = monthly_ds.values[mask]
da2_clean_1 = mld_mean.values[mask]
std_clean_1 = mld_std.values[mask]  # 新增：提取对应时段有效标准差
times_clean_1 = monthly_ds['time'].values[mask]

# TAO

In [ ]:
ds1 = xr.open_dataset(configured_path('tao_temperature_file'))
ds1 = ds1.T_20.sel(lat=slice(-8.5,8.5),time=slice('1987-10','2023-12'))
#ds1

In [ ]:
ds2 = xr.open_dataset(configured_path('tao_salinity_file'))
ds2 = ds2.S_41.sel(time=slice('1987-10','2023-12'))
#ds2

In [ ]:
import numpy as np
import xarray as xr
from scipy.interpolate import interp1d

T_20 = ds1
depth_old = T_20.depth.values
depth_new = np.array([5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,
                      125,150,175,200,225,250,275,300,325,350,375,400,425,450,475,500,
                      550,600,650,700,750])

lon = T_20.lon.values
lat = T_20.lat.values
time = T_20.time.values

interp_data = np.full((len(lon), len(lat), len(depth_new), len(time)), np.nan, dtype=T_20.dtype)

for i_lon in range(len(lon)):
    for j_lat in range(len(lat)):
        for k_time in range(len(time)):
            profile = T_20.values[i_lon, j_lat, :, k_time]
            valid = ~np.isnan(profile)
            if np.sum(valid) < 2:
                continue  # 不足以插值，保持nan
            f = interp1d(depth_old[valid], profile[valid], bounds_error=False, fill_value=np.nan)
            interp_data[i_lon, j_lat, :, k_time] = f(depth_new)

T_20_interp = xr.DataArray(interp_data,
                          dims=['lon','lat','depth','time'],
                          coords={'lon': lon, 'lat': lat, 'depth': depth_new, 'time': time})

sa = ds2
depth_old = sa.depth.values
depth_new = np.array([5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100,
                      125,150,175,200,225,250,275,300,325,350,375,400,425,450,475,500,
                      550,600,650,700,750])

lon = sa.lon.values
lat = sa.lat.values
time = sa.time.values

interp_data = np.full((len(lon), len(lat), len(depth_new), len(time)), np.nan, dtype=T_20.dtype)

for i_lon in range(len(lon)):
    for j_lat in range(len(lat)):
        for k_time in range(len(time)):
            profile = sa.values[i_lon, j_lat, :, k_time]
            valid = ~np.isnan(profile)
            if np.sum(valid) < 2:
                continue  # 不足以插值，保持nan
            f = interp1d(depth_old[valid], profile[valid], bounds_error=False, fill_value=np.nan)
            interp_data[i_lon, j_lat, :, k_time] = f(depth_new)

s_interp = xr.DataArray(interp_data,
                          dims=['lon','lat','depth','time'],
                          coords={'lon': lon, 'lat': lat, 'depth': depth_new, 'time': time})

In [ ]:
def compute_mld_sigma0_vectorized(ds, temp_name='thetao_cglo', salt_name='so_cglo',
                                  depth_name='depth', lat_name='lat', lon_name='lon',
                                  threshold=0.03, ref_depth=10, show_progress=True):
    """
    向量化计算混合层深度 MLD (|Δσ₀|=threshold kg/m³, 参考深度 ref_depth)
    支持数据维度: (time, depth, lat, lon)
    第一个超过阈值的深度必须 > ref_depth
    """
    temp = ds[temp_name].values  # (time, depth, lat, lon)
    salt = ds[salt_name].values
    depth = ds[depth_name].values
    lat = ds[lat_name].values
    lon = ds[lon_name].values
    time = ds["time"].values

    ntime, ndepth, nlat, nlon = temp.shape
    npoints = nlat * nlon

    # 展平空间维度
    theta_flat = temp.reshape(ntime, ndepth, npoints)
    salt_flat  = salt.reshape(ntime, ndepth, npoints)
    lat_flat   = np.repeat(lat, nlon)
    lon_flat   = np.tile(lon, nlat)

    mld = np.full((ntime, npoints), np.nan)

    iterator = range(ntime)
    if show_progress:
        iterator = tqdm(iterator, desc="Computing MLD (daily)")

    for t in iterator:
        theta_t = theta_flat[t]  # (depth, points)
        salt_t  = salt_flat[t]

        # 掩码有效数据
        mask = np.isfinite(theta_t) & np.isfinite(salt_t)
        valid_points = np.any(mask, axis=0)
        if not np.any(valid_points):
            continue

        theta_valid = np.where(mask, theta_t, np.nan)
        salt_valid  = np.where(mask, salt_t, np.nan)

        # 批量计算压力、盐度、密度
        p = gsw.p_from_z(-depth[:, None], lat_flat[None, :])
        SA = gsw.SA_from_SP(salt_valid, p, lon_flat[None, :], lat_flat[None, :])
        sigma0 = gsw.sigma0(SA, theta_valid)  # (depth, points)

        # 参考深度密度 (每个剖面单独插值)
        sigma0_ref = np.apply_along_axis(
            lambda prof: np.interp(ref_depth, depth, prof, left=np.nan, right=np.nan),
            axis=0,
            arr=sigma0
        )

        delta = np.abs(sigma0 - sigma0_ref[None, :])

        # 找到超过阈值且深度 > ref_depth 的第一个索引
        over_mask = (delta >= threshold) & (depth[:, None] > ref_depth)
        over_idx = np.argmax(over_mask, axis=0)
        over_idx[np.all(over_mask == False, axis=0)] = -1  # 没有满足条件的点标记为 -1

        # 批量线性插值
        d1 = np.where(over_idx>0, depth[over_idx-1], np.nan)
        d2 = np.where(over_idx>0, depth[over_idx], np.nan)
        r1 = np.where(over_idx>0, delta[over_idx-1, np.arange(npoints)], np.nan)
        r2 = np.where(over_idx>0, delta[over_idx, np.arange(npoints)], np.nan)
        frac = (threshold - r1) / (r2 - r1)
        mld[t] = d1 + frac*(d2 - d1)
        mld[t, over_idx==-1] = np.nan

    # reshape 回 (time, lat, lon)
    mld = mld.reshape(ntime, nlat, nlon)
    mld_da = xr.DataArray(
        mld,
        coords={"time": time, "lat": lat, "lon": lon},
        dims=("time", "lat", "lon"),
        name=f"MLD_sigma0_{threshold*1000:03.0f}",
        attrs={
            "long_name": f"Mixed Layer Depth (|Δσ₀|={threshold} kg/m³, ref={ref_depth} m)",
            "units": "m",
            "method": "TEOS-10 σ₀ threshold",
        }
    )
    return mld_da

In [ ]:
# 假设原来是 (depth, lat, lon, time)，转换成 (time, depth, lat, lon)
t_interp = T_20_interp.transpose('time', 'depth', 'lat', 'lon')
s_interp = s_interp.transpose('time', 'depth', 'lat', 'lon')


ds = xr.Dataset({
    "thetao_cglo": t_interp,
    "so_cglo": s_interp
})
mld_da = compute_mld_sigma0_vectorized(ds)


In [ ]:
# 统计每个格点时间维度非NaN数量
non_nan_count = mld_da.count(dim='time')

# 展平成一维数组，方便排序
count_flat = non_nan_count.values.flatten()
lat_len, lon_len = non_nan_count.shape

# 1. 获取从大到小排序的一维索引（前3个）
top3_1d_idx = np.argsort(count_flat)[::-1][:5]

# 2. 循环解析每个点二维索引、坐标、计数值
result = []
for idx_1d in top3_1d_idx:
    lat_idx, lon_idx = np.unravel_index(idx_1d, (lat_len, lon_len))
    cnt_val = count_flat[idx_1d]
    lat_val = non_nan_count.lat.values[lat_idx]
    lon_val = non_nan_count.lon.values[lon_idx]
    result.append({
        "count": cnt_val,
        "lat": lat_val,
        "lon": lon_val
    })

# 打印输出
for i, point in enumerate(result, 1):
    print(f"第{i}名：非NaN数量={point['count']}, 纬度={point['lat']:.2f}, 经度={point['lon']:.2f}")

In [ ]:
lat0, lon0 = 8, 137

fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=15)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)

mld_hot = mld_da.sel(lat=lat0, lon=lon0,time=slice('2010-1','2020-12'))
mld_mean, mld_std = four_neighbor_stats(ds, "MLD_pred", lat0, lon0)

mld_mean = mld_mean.sel(time=slice('2010-1','2020-12'))
mld_std = mld_std.sel(time=slice('2010-1','2020-12'))

# 【修改2】同步筛选std，保证时间对齐
da1_clean_2 = mld_hot.values
da2_clean_2 = mld_mean.values
std_clean_2 = mld_std.values
times_clean_2 = mld_hot['time'].values

# ALOHA

In [ ]:
ds1 = xr.open_dataset(str(Path(configured_path('work_dir')) / 'HOT_MLD_clean.nc'))
#ds1

In [ ]:
# 1. 按月初重采样求平均
monthly_ds = ds1.MLD_mean.resample(time='1MS').mean()

# 2. 将时间坐标向后移动15天，使时间点落在月中
monthly_ds['time'] = monthly_ds['time'] + pd.Timedelta(days=14)

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=14)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)

lat0, lon0 = 22.75, 202
mld_mean,mld_std = four_neighbor_stats(ds, "MLD_pred", lat0, lon0)
#mld_interp

In [ ]:
monthly_ds = monthly_ds.sel(time=slice('2010-1','2020-12'))
mld_std = mld_std.sel(time=slice('2010-1','2020-12'))
mld_mean = mld_mean.sel(time=slice('2010-1','2020-12'))

# 【修改1】mask第二个条件改为 mld_mean，原mld_pred变量不存在
mask = (~np.isnan(monthly_ds)) & (~np.isnan(mld_mean))
da1_clean_3 = monthly_ds.values[mask]
da2_clean_3 = mld_mean.values[mask]
std_clean_3 = mld_std.values[mask]
times_clean_3 = monthly_ds['time'].values[mask]

# Papa

In [ ]:
ds1 = xr.open_dataset(configured_path('papa_density_file'))
ds1 = ds1.where(ds1 != 1e35, np.nan)
ds1

In [ ]:
import numpy as np
import xarray as xr
from tqdm import tqdm

def compute_mld_from_density(ds, density_name='sigma0',
                             depth_name='depth', lat_name='lat', lon_name='lon',
                             threshold=0.03, ref_depth=10, show_progress=True):
    """
    计算混合层深度 MLD，基于已有密度数据 sigma0，阈值为 threshold kg/m³，参考深度为 ref_depth
    输入数据维度：(time, depth, lat, lon)
    """
    density = ds[density_name].values  # (time, depth, lat, lon)
    depth = ds[depth_name].values
    lat = ds[lat_name].values
    lon = ds[lon_name].values
    time = ds["time"].values

    ntime, ndepth, nlat, nlon = density.shape
    npoints = nlat * nlon

    density_flat = density.reshape(ntime, ndepth, npoints)
    mld = np.full((ntime, npoints), np.nan)

    iterator = range(ntime)
    if show_progress:
        iterator = tqdm(iterator, desc="Computing MLD from density")

    for t in iterator:
        dens_t = density_flat[t]  # (depth, points)

        # 有效数据掩码（排除nan）
        mask = np.isfinite(dens_t)
        valid_points = np.any(mask, axis=0)
        if not np.any(valid_points):
            continue

        dens_valid = np.where(mask, dens_t, np.nan)

        # 参考深度密度，插值获取
        rho_ref = np.apply_along_axis(
            lambda prof: np.interp(ref_depth, depth, prof, left=np.nan, right=np.nan),
            axis=0,
            arr=dens_valid
        )

        delta = np.abs(dens_valid - rho_ref[None, :])

        over_mask = (delta >= threshold) & (depth[:, None] > ref_depth)
        over_idx = np.argmax(over_mask, axis=0)
        no_mld = ~np.any(over_mask, axis=0)
        over_idx[no_mld] = -1

        d1 = np.where(over_idx > 0, depth[over_idx - 1], np.nan)
        d2 = np.where(over_idx > 0, depth[over_idx], np.nan)
        r1 = np.where(over_idx > 0, delta[over_idx - 1, np.arange(npoints)], np.nan)
        r2 = np.where(over_idx > 0, delta[over_idx, np.arange(npoints)], np.nan)

        frac = (threshold - r1) / (r2 - r1)
        mld[t] = d1 + frac * (d2 - d1)

        mld[t, over_idx == -1] = np.nan

    mld = mld.reshape(ntime, nlat, nlon)

    mld_da = xr.DataArray(
        mld,
        coords={"time": time, "lat": lat, "lon": lon},
        dims=("time", "lat", "lon"),
        name=f"MLD_sigma0_{int(threshold*1000):03d}",
        attrs={
            "long_name": f"Mixed Layer Depth (|Δσ₀|={threshold} kg/m³, ref={ref_depth} m)",
            "units": "m",
            "method": "Density threshold method",
        }
    )
    return mld_da

In [ ]:
mld_da = compute_mld_from_density(ds1, density_name='STH_71', threshold=0.03, ref_depth=10)

In [ ]:
fdir1 = configured_path('paired_mld_file')
ds = xr.open_mfdataset(fdir1)

time_new = pd.date_range(
    start='1982-01-1',
    periods=ds.sizes['time'],
    freq='MS'  # 每月开始
) + pd.Timedelta(days=15)  # 调整到每月中旬
ds = ds.assign_coords(time=time_new)

lat0, lon0 = 50.1, 215.1
mld_mean,mld_std = four_neighbor_stats(ds, "MLD_pred", lat0, lon0)
#mld_interp

In [ ]:
mld_da = mld_da.isel(lat=0,lon=0)
monthly_ds = mld_da.sel(time=slice('2010-1','2020-12'))
mld_mean = mld_mean.sel(time=slice('2010-1','2020-12'))
mld_std = mld_std.sel(time=slice('2010-1','2020-12'))

mask = (~np.isnan(monthly_ds.values)) & (~np.isnan(mld_mean.values))

da1_clean_4 = monthly_ds.values[mask]
da2_clean_4 = mld_mean.values[mask]
std_clean_4 = mld_std.values[mask]
times_clean_4 = monthly_ds['time'].values[mask]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import font_manager
from pathlib import Path

font_dir = Path(configured_path('font_dir'))
font_files = [font_dir / 'Times New Roman.ttf', font_dir / 'Times New Roman Bold.ttf']
for font_file in font_files:
    if not font_file.is_file():
        raise FileNotFoundError(f'Required Times New Roman font not found: {font_file}')
    font_manager.fontManager.addfont(str(font_file))
resolved_font = Path(font_manager.findfont('Times New Roman', fallback_to_default=False)).resolve()
if resolved_font not in {font.resolve() for font in font_files}:
    raise RuntimeError(f'Times New Roman resolved to an unexpected file: {resolved_font}')

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'custom',
    'mathtext.rm': 'Times New Roman',
    'mathtext.it': 'Times New Roman:italic',
    'mathtext.bf': 'Times New Roman:bold',
    'axes.unicode_minus': False,
})

SOTS_FILE = configured_path('sots_mld_file')
PIRATA_FILE = configured_path('pirata_mld_file')
PRED_FILE = configured_path('paired_mld_file')

pred_ds = xr.open_dataset(PRED_FILE)
pred_time = pd.date_range('1982-01-01', periods=pred_ds.sizes['time'], freq='MS') + pd.Timedelta(days=14)
if pred_ds.sizes['time'] != len(pred_time):
    raise ValueError(f'Unexpected prediction time length: {pred_ds.sizes["time"]}')
pred_ds = pred_ds.assign_coords(time=pred_time)

def prepare_monthly_station(station_file, target_lat, target_lon):
    with xr.open_dataset(station_file) as station_ds:
        obs = station_ds['MLD_monthly_mean'].sel(time=slice('2010-01', '2020-12')).load()
    obs_time = pd.DatetimeIndex(obs.time.values).to_period('M').to_timestamp() + pd.Timedelta(days=14)
    obs = obs.assign_coords(time=obs_time)
    pred, pred_std = four_neighbor_stats(pred_ds, 'MLD_pred', target_lat, target_lon)
    pred = pred.sel(time=slice('2010-01', '2020-12')).load()
    pred_std = pred_std.sel(time=slice('2010-01', '2020-12')).load()
    obs, pred, pred_std = xr.align(obs, pred, pred_std, join='inner')
    valid = np.isfinite(obs.values) & np.isfinite(pred.values) & np.isfinite(pred_std.values)
    return obs.values[valid], pred.values[valid], obs.time.values[valid]

da1_clean_5, da2_clean_5, times_clean_5 = prepare_monthly_station(SOTS_FILE, -47.0, 142.0)
with xr.open_dataset(PIRATA_FILE) as pirata_meta:
    pirata_lat = float(pirata_meta['lat'].values)
    pirata_lon = float(pirata_meta['lon'].values) % 360
da1_clean_6, da2_clean_6, times_clean_6 = prepare_monthly_station(
    PIRATA_FILE, pirata_lat, pirata_lon
)
pred_ds.close()

def compute_monthly_climatology(data, times):
    monthly_means = np.full(12, np.nan)
    monthly_stds = np.full(12, np.nan)
    monthly_sems = np.full(12, np.nan)
    months = np.array([t.astype('datetime64[M]').astype(int) % 12 + 1 for t in times])
    for m in range(1, 13):
        vals = np.asarray(data[months == m], dtype=float)
        vals = vals[np.isfinite(vals)]
        if vals.size > 0:
            monthly_means[m-1] = np.nanmean(vals)
        if vals.size > 1:
            monthly_stds[m-1] = np.nanstd(vals, ddof=1)
            monthly_sems[m-1] = monthly_stds[m-1] / np.sqrt(vals.size)
    return monthly_means, monthly_stds, monthly_sems

da1_clean_list = [da1_clean_1, da1_clean_2, da1_clean_3, da1_clean_4, da1_clean_5, da1_clean_6]
da2_clean_list = [da2_clean_1, da2_clean_2, da2_clean_3, da2_clean_4, da2_clean_5, da2_clean_6]
times_clean_list = [times_clean_1, times_clean_2, times_clean_3, times_clean_4, times_clean_5, times_clean_6]
    
titles = ['BAT (31.67°N, 295.83°E)', 'TAO (8°N, 137°E)',
          'HOT (22.75°N, 202°E)', 'Papa (50.13°N, 215.17°E)',
          'SOTS (47°S, 142°E)', 'PIRATA (10°S, 350°E)']
labels = ['BAT', 'TAO', 'HOT', 'Papa', 'SOTS', 'PIRATA']
subplot_labels = ['(a)', '(b)', '(c)', '(d)', '(e)', '(f)']

fig, axes = plt.subplots(2, 3, figsize=(22.5, 10), sharex='col')
axes = axes.flatten()
months = np.arange(1, 13)
uncertainty = 'sem'  # Use 'std' for interannual spread instead.

for i in range(6):
    true_clim, true_std, true_sem = compute_monthly_climatology(
        da1_clean_list[i], times_clean_list[i]
    )
    pred_clim, pred_std, pred_sem = compute_monthly_climatology(
        da2_clean_list[i], times_clean_list[i]
    )
    true_spread = true_sem if uncertainty == 'sem' else true_std
    pred_spread = pred_sem if uncertainty == 'sem' else pred_std

    ax = axes[i]
    ax.fill_between(months, true_clim - true_spread, true_clim + true_spread,
                    color='#1f77b4', alpha=0.16, linewidth=0)
    ax.fill_between(months, pred_clim - pred_spread, pred_clim + pred_spread,
                    color='#d62728', alpha=0.16, linewidth=0)
    ax.plot(months, true_clim, marker='o', linewidth=3.0,
            color='#1f77b4', label=labels[i])
    ax.plot(months, pred_clim, marker='x', linewidth=3.0,
            color='#d62728', label='SJTU-ViT')

    if i >= 3:
        ax.set_xlabel('Month', fontsize=21)

    if i % 3 == 0:
        ax.set_ylabel('Climatology MLD (m)', fontsize=21)

    ax.set_title(titles[i], fontsize=24)
    ax.tick_params(axis='both', which='major', labelsize=21)
    if i < 3:
        ax.tick_params(axis='x', which='both', bottom=True, labelbottom=False)

    ax.legend(fontsize=18)
    #ax.grid(True)
    
    ax.text(0.02, 1.11, subplot_labels[i], transform=ax.transAxes,
            fontsize=26,  va='top', ha='left')

plt.tight_layout()
plt.show()


In [ ]:
# Alternative version: monthly climatology with ±1 interannual STD shading
fig, axes = plt.subplots(2, 3, figsize=(22.5, 10), sharex='col')
axes = axes.flatten()
months = np.arange(1, 13)

for i in range(6):
    true_clim, true_std, _ = compute_monthly_climatology(
        da1_clean_list[i], times_clean_list[i]
    )
    pred_clim, pred_std, _ = compute_monthly_climatology(
        da2_clean_list[i], times_clean_list[i]
    )

    ax = axes[i]
    ax.fill_between(months, true_clim - true_std, true_clim + true_std,
                    color='#1f77b4', alpha=0.16, linewidth=0)
    ax.fill_between(months, pred_clim - pred_std, pred_clim + pred_std,
                    color='#d62728', alpha=0.16, linewidth=0)
    ax.plot(months, true_clim, marker='o', linewidth=3.0,
            color='#1f77b4', label=labels[i])
    ax.plot(months, pred_clim, marker='x', linewidth=3.0,
            color='#d62728', label='SJTU-ViT')

    if i >= 3:
        ax.set_xlabel('Month', fontsize=21)
    if i % 3 == 0:
        ax.set_ylabel('Climatology MLD (m)', fontsize=21)

    ax.set_title(titles[i], fontsize=24)
    ax.tick_params(axis='both', which='major', labelsize=21)
    if i < 3:
        ax.tick_params(axis='x', which='both', bottom=True, labelbottom=False)

    ax.legend(fontsize=18)
    ax.text(0.02, 1.11, subplot_labels[i], transform=ax.transAxes,
            fontsize=26, va='top', ha='left')

plt.tight_layout()
output_dir = Path(configured_path('output_dir')).resolve()
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 's5figure.png'
fig.savefig(output_path, dpi=600, bbox_inches='tight', pad_inches=0)
print(f'Figure saved to: {output_path}')
plt.show()
